# 01 - Recommendation Model Training (Two-Tower DDP)

Uses `CustomTrainer(func=train_fn)` with the built-in `torch-distributed` ClusterTrainingRuntime.
The controller automatically injects `PET_*` env vars and wraps with `torchrun`.

| Step | What happens |
|------|-------------|
| 1 | SDK serializes `train_fn` → TrainJob CR |
| 2 | Controller creates JobSet (N pods, torchrun, PET_* envs) |
| 3 | PyTorch DDP trains across nodes/GPUs |
| 4 | Rank-0 logs to MLflow + saves model to S3 |

In [ ]:
%pip install -q kubeflow --no-cache-dir \
    --index-url https://console.redhat.com/api/pypi/public-rhai/rhoai/3.3/cuda12.9-ubi9/simple/
%pip install -q kubernetes "fsspec[s3]" s3fs boto3 scikit-learn

In [ ]:
import os

# Papermill parameters — overridden at runtime via -p flags (values come from .env)
NAMESPACE = os.environ.get("NAMESPACE", "smartshop")
RUNTIME = "torch-distributed"
DATA_DIR = os.environ.get("DATA_DIR", f"s3://{os.environ.get('S3_FEATURES_BUCKET', 'smartshop-features')}")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", f"s3://{os.environ.get('S3_MODELS_BUCKET', 'smartshop-models')}/recommendation")
MAX_ROWS = int(os.environ.get("REC_MAX_ROWS", "5000000"))
EPOCHS = int(os.environ.get("REC_TRAIN_EPOCHS", "15"))
BATCH_SIZE = int(os.environ.get("REC_TRAIN_BATCH_SIZE", "2048"))
LR = 0.0003
EMBED_DIM = int(os.environ.get("REC_EMBED_DIM", "64"))
HIDDEN_DIM = int(os.environ.get("REC_HIDDEN_DIM", "256"))
NUM_NODES = int(os.environ.get("REC_TRAIN_NODES", "4"))
GPUS_PER_NODE = int(os.environ.get("REC_GPUS_PER_NODE", "2"))
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", os.environ.get("AWS_ENDPOINT_URL_S3", ""))
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "")
CLUSTER_DOMAIN = os.environ.get("OC_CLUSTER_DOMAIN", "")
TIMEOUT_SECONDS = 7200
S3_CREDENTIALS_SECRET = "smartshop-credentials"
MLFLOW_SECRET = "smartshop-mlflow-token"

## Authentication

In [ ]:
import os
import warnings
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", message=".*Unverified HTTPS.*")

IN_CLUSTER = os.path.exists("/var/run/secrets/kubernetes.io/serviceaccount/token")
K8S_TOKEN = os.getenv("K8S_TOKEN", "")
K8S_API = os.getenv("K8S_API_SERVER", "")

if not K8S_TOKEN and IN_CLUSTER:
    with open("/var/run/secrets/kubernetes.io/serviceaccount/token") as f:
        K8S_TOKEN = f.read().strip()
    K8S_API = "https://kubernetes.default.svc"

if not K8S_API and CLUSTER_DOMAIN:
    K8S_API = f"https://api.{CLUSTER_DOMAIN.replace('apps.', '')}:6443"

print(f"K8S API: {K8S_API or '(in-cluster)'}")
print(f"Token: {K8S_TOKEN[:20]}..." if K8S_TOKEN else "No token — using in-cluster defaults")
print(f"In-cluster: {IN_CLUSTER}")

## Initialize Kubeflow TrainerClient

In [ ]:
from kubernetes import client as k8s
from kubeflow.trainer import TrainerClient
from kubeflow.common.types import KubernetesBackendConfig

cfg = None
if K8S_TOKEN:
    cfg = k8s.Configuration()
    if K8S_API:
        cfg.host = K8S_API
    cfg.verify_ssl = False
    cfg.api_key = {"authorization": f"Bearer {K8S_TOKEN}"}

trainer = TrainerClient(
    KubernetesBackendConfig(
        namespace=NAMESPACE,
        client_configuration=cfg
    )
)

runtime = trainer.get_runtime(RUNTIME)
print(f"Runtime: {RUNTIME}")

## Define Training Function

Self-contained function that runs DDP training. The SDK serializes this into the pod.
`torchrun` + `PET_*` env vars are injected automatically by the controller.

In [ ]:
def train_fn(
    data_dir: str,
    output_dir: str,
    epochs: int = 10,
    batch_size: int = 2048,
    lr: float = 3e-4,
    embed_dim: int = 64,
    hidden_dim: int = 256,
    max_rows: int = 5000000,
):
    """Two-Tower recommendation model DDP training.

    Binary classification: rating >= 4 is positive (liked).
    Saves full checkpoint with ID mappings for serving.
    torchrun + PET_* env vars are injected by the Kubeflow Trainer controller.
    """
    import os
    import io
    import json

    import pandas as pd
    import numpy as np
    import torch
    import torch.nn as nn
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader, TensorDataset, DistributedSampler
    import fsspec
    import mlflow

    # DDP setup (torchrun sets RANK, LOCAL_RANK, WORLD_SIZE)
    dist.init_process_group(backend="nccl")
    rank = dist.get_rank()
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = dist.get_world_size()
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)

    if rank == 0:
        print(f"DDP initialized: world_size={world_size}, device={device}")
        print(f"PET_NNODES={os.environ.get('PET_NNODES')}, PET_NPROC_PER_NODE={os.environ.get('PET_NPROC_PER_NODE')}")

    # MLflow init (rank 0 only)
    import time as _time
    import logging
    logging.getLogger("mlflow.tracing.export.mlflow_v3").setLevel(logging.ERROR)
    use_mlflow = False
    patience = 4
    if rank == 0:
        try:
            os.environ.setdefault("MLFLOW_TRACKING_INSECURE_TLS", "true")
            workspace = os.environ.pop("MLFLOW_WORKSPACE", None)
            tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "").rstrip("/")
            if tracking_uri.endswith("/mlflow"):
                tracking_uri = tracking_uri[:-len("/mlflow")]
                os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
            if tracking_uri:
                mlflow.set_tracking_uri(tracking_uri)
            if workspace:
                from mlflow.utils import rest_utils as _ru
                _orig_http = _ru.http_request
                def _ws_http(*a, **kw):
                    h = kw.get("extra_headers", {}) or {}
                    h["X-MLflow-Workspace"] = workspace
                    kw["extra_headers"] = h
                    return _orig_http(*a, **kw)
                _ru.http_request = _ws_http
            mlflow.set_experiment("smartshop-rec-training")
            mlflow.start_run(run_name=f"twotower-{world_size}gpu-adamw-warmup",
                             description=f"Two-Tower collaborative filtering | {world_size}x A100-80GB | "
                                         f"AdamW+warmup+cosine | grad_clip=1.0 | "
                                         f"neg_ratio=3 | bs={batch_size} | {epochs} epochs")
            use_mlflow = True
        except Exception as e:
            print(f"MLflow early init failed (non-fatal): {e}")

    # Load data from S3
    s3_endpoint = os.environ.get("AWS_ENDPOINT_URL_S3", "")
    storage_options = {"endpoint_url": s3_endpoint} if s3_endpoint else {}
    fs, _ = fsspec.core.url_to_fs(data_dir, **storage_options)

    if rank == 0:
        print(f"Loading interactions from {data_dir}/interactions/...")
    
    parquet_path = f"{data_dir}/interactions"
    files = [f for f in fs.ls(parquet_path) if f.endswith(".parquet")]
    
    dfs = []
    rows_so_far = 0
    for f in sorted(files):
        with fs.open(f, "rb") as fh:
            chunk = pd.read_parquet(io.BytesIO(fh.read()))
        dfs.append(chunk)
        rows_so_far += len(chunk)
        if max_rows > 0 and rows_so_far >= max_rows:
            break
    
    df = pd.concat(dfs, ignore_index=True)
    if max_rows > 0:
        df = df.head(max_rows)
    if rank == 0:
        print(f"Loaded {len(df)} interactions")

    # Encode user/item IDs
    user_ids = df["user_id"].astype("category")
    item_ids = df["item_id"].astype("category")
    n_users = user_ids.cat.categories.size
    n_items = item_ids.cat.categories.size

    user_to_idx = dict(zip(user_ids.cat.categories.tolist(), range(n_users)))
    item_to_idx = dict(zip(item_ids.cat.categories.tolist(), range(n_items)))

    user_idx = torch.tensor(user_ids.cat.codes.values, dtype=torch.long)
    item_idx = torch.tensor(item_ids.cat.codes.values, dtype=torch.long)
    labels = torch.tensor((df["rating"].values >= 4).astype(np.float32), dtype=torch.float32)

    # Negative sampling: add random user-item pairs not in data as negatives
    neg_ratio = 3
    n_neg = len(df) * neg_ratio
    neg_users = torch.tensor(np.random.randint(0, n_users, size=n_neg), dtype=torch.long)
    neg_items = torch.tensor(np.random.randint(0, n_items, size=n_neg), dtype=torch.long)
    neg_labels = torch.zeros(n_neg, dtype=torch.float32)

    all_users = torch.cat([user_idx, neg_users])
    all_items = torch.cat([item_idx, neg_items])
    all_labels = torch.cat([labels, neg_labels])

    if rank == 0:
        pos_rate = all_labels.mean().item()
        print(f"n_users={n_users}, n_items={n_items}, samples={len(all_labels):,} "
              f"(real={len(df):,} + neg={n_neg:,}), positive_rate={pos_rate:.3f}")

    # Train/val split (90/10)
    n_total = len(all_labels)
    indices = np.random.permutation(n_total)
    split = int(0.9 * n_total)
    train_indices, val_indices = indices[:split], indices[split:]

    train_ds = TensorDataset(all_users[train_indices], all_items[train_indices], all_labels[train_indices])
    val_ds = TensorDataset(all_users[val_indices], all_items[val_indices], all_labels[val_indices])

    train_sampler = DistributedSampler(train_ds, num_replicas=world_size, rank=rank, shuffle=True)
    val_sampler = DistributedSampler(val_ds, num_replicas=world_size, rank=rank, shuffle=False)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=train_sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, sampler=val_sampler, num_workers=2, pin_memory=True)

    # Two-Tower Model
    class TwoTower(nn.Module):
        def __init__(self, n_users, n_items, embed_dim, hidden_dim):
            super().__init__()
            self.user_embed = nn.Embedding(n_users, embed_dim)
            self.item_embed = nn.Embedding(n_items, embed_dim)
            self.user_mlp = nn.Sequential(
                nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(hidden_dim, embed_dim),
            )
            self.item_mlp = nn.Sequential(
                nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(hidden_dim, embed_dim),
            )

        def forward(self, users, items):
            u = self.user_mlp(self.user_embed(users))
            i = self.item_mlp(self.item_embed(items))
            return (u * i).sum(dim=1)

    model = TwoTower(n_users, n_items, embed_dim, hidden_dim).to(device)
    model = DDP(model, device_ids=[local_rank])

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    warmup_epochs = max(1, epochs // 10)
    warmup_sched = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs, eta_min=lr * 0.01)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup_sched, cosine_sched], milestones=[warmup_epochs])
    criterion = nn.BCEWithLogitsLoss()
    GRAD_CLIP_NORM = 1.0

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # MLflow params logging
    if rank == 0 and use_mlflow:
        try:
            gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
            gpu_mem_gb = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0
            mlflow.log_params({
                "model": "TwoTower",
                "n_users": n_users, "n_items": n_items,
                "embed_dim": embed_dim, "hidden_dim": hidden_dim,
                "dropout": 0.2,
                "batch_size": batch_size, "lr": lr,
                "optimizer": "AdamW",
                "weight_decay": 1e-5,
                "grad_clip_norm": GRAD_CLIP_NORM,
                "lr_scheduler": "LinearWarmup+CosineAnnealing",
                "warmup_epochs": warmup_epochs,
                "early_stopping_patience": patience,
                "world_size": world_size,
                "num_nodes": int(os.environ.get("PET_NNODES", 1)) if os.environ.get("PET_NNODES", "1").isdigit() else world_size // torch.cuda.device_count(),
                "gpus_per_node": torch.cuda.device_count(),
                "gpu_type": gpu_name, "gpu_mem_gb": gpu_mem_gb,
                "epochs": epochs, "max_rows": max_rows,
                "loss_fn": "BCEWithLogitsLoss",
                "total_params": total_params,
                "trainable_params": trainable_params,
                "train_samples": len(train_ds),
                "val_samples": len(val_ds),
                "positive_rate": f"{all_labels.mean().item():.3f}",
                "neg_sampling_ratio": neg_ratio,
                "data_source": data_dir,
                "output_dir": output_dir,
            })
            mlflow.set_tags({
                "framework": "pytorch",
                "distributed": "DDP",
                "platform": "Red Hat OpenShift AI",
                "task": "recommendation",
            })
        except Exception as e:
            print(f"MLflow params logging failed (non-fatal): {e}")

    train_start = _time.time()

    if rank == 0 and use_mlflow:
        try:
            with mlflow.start_span(name="data_loading") as _sp:
                _sp.set_inputs({"data_dir": data_dir, "max_rows": max_rows})
                _sp.set_outputs({"n_users": n_users, "n_items": n_items,
                                 "total_samples": len(all_labels), "train_samples": len(train_ds),
                                 "val_samples": len(val_ds), "positive_rate": f"{all_labels.mean().item():.3f}"})
            with mlflow.start_span(name="model_init") as _sp:
                _sp.set_inputs({"embed_dim": embed_dim, "hidden_dim": hidden_dim})
                _sp.set_outputs({"total_params": total_params, "trainable_params": trainable_params,
                                 "optimizer": "AdamW", "scheduler": "LinearWarmup+CosineAnnealing",
                                 "gpu_type": gpu_name, "gpu_mem_gb": gpu_mem_gb})
        except Exception:
            pass

    best_val_loss = float("inf")
    patience_counter = 0
    for epoch in range(epochs):
        epoch_start = _time.time()
        train_sampler.set_epoch(epoch)
        model.train()
        epoch_loss = 0.0
        n_samples = 0
        _global_step = epoch * len(train_loader)
        _log_interval = max(1, len(train_loader) // 20)
        for _batch_idx, (users, items, targets) in enumerate(train_loader):
            users, items, targets = users.to(device), items.to(device), targets.to(device)
            optimizer.zero_grad()
            preds = model(users, items)
            loss = criterion(preds, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step()
            epoch_loss += loss.item() * len(users)
            n_samples += len(users)
            _global_step += 1
            if rank == 0 and use_mlflow and _batch_idx % _log_interval == 0:
                mlflow.log_metric("step_loss", loss.item(), step=_global_step)

        avg_train_loss = epoch_loss / n_samples
        scheduler.step()

        # Validation
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        all_preds_val = []
        all_targets_val = []
        with torch.no_grad():
            for users, items, targets in val_loader:
                users, items, targets = users.to(device), items.to(device), targets.to(device)
                preds = model(users, items)
                val_loss += criterion(preds, targets).item() * len(users)
                val_correct += ((preds > 0).float() == targets).sum().item()
                val_total += len(users)
                all_preds_val.append(torch.sigmoid(preds).cpu())
                all_targets_val.append(targets.cpu())

        avg_val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        all_preds_val = torch.cat(all_preds_val).numpy()
        all_targets_val = torch.cat(all_targets_val).numpy()
        from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
        val_auc = roc_auc_score(all_targets_val, all_preds_val)
        val_prec = precision_score(all_targets_val, (all_preds_val > 0.5).astype(int), zero_division=0)
        val_recall = recall_score(all_targets_val, (all_preds_val > 0.5).astype(int), zero_division=0)
        val_f1 = f1_score(all_targets_val, (all_preds_val > 0.5).astype(int), zero_division=0)
        epoch_time = _time.time() - epoch_start
        throughput = n_samples * world_size / epoch_time
        current_lr = scheduler.get_last_lr()[0]

        if rank == 0:
            print(f"Epoch {epoch+1}/{epochs} train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f} "
                  f"val_acc={val_acc:.4f} val_auc={val_auc:.4f} val_f1={val_f1:.4f} "
                  f"lr={current_lr:.2e} time={epoch_time:.1f}s throughput={throughput:,.0f} samples/s")
            if use_mlflow:
                mlflow.log_metrics({
                    "train_loss": avg_train_loss,
                    "val_loss": avg_val_loss,
                    "val_acc": val_acc,
                    "val_auc_roc": val_auc,
                    "val_precision": val_prec,
                    "val_recall": val_recall,
                    "val_f1": val_f1,
                    "learning_rate": current_lr,
                    "epoch_time_s": epoch_time,
                    "throughput_samples_per_sec": throughput,
                }, step=epoch)
                try:
                    with mlflow.start_span(name=f"epoch_{epoch+1}") as _sp:
                        _sp.set_inputs({"epoch": epoch + 1, "lr": current_lr})
                        _sp.set_outputs({"train_loss": avg_train_loss, "val_loss": avg_val_loss,
                                         "val_acc": val_acc, "val_auc_roc": val_auc,
                                         "epoch_time_s": round(epoch_time, 1),
                                         "throughput": round(throughput)})
                except Exception:
                    pass

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                patience_counter = 0
                checkpoint = {
                    "model_state_dict": model.module.state_dict(),
                    "n_users": n_users,
                    "n_items": n_items,
                    "embed_dim": embed_dim,
                    "hidden_dim": hidden_dim,
                    "user_to_idx": user_to_idx,
                    "item_to_idx": item_to_idx,
                    "best_val_loss": best_val_loss,
                    "best_val_acc": val_acc,
                    "best_val_auc_roc": val_auc,
                    "best_val_f1": val_f1,
                    "epoch": epoch + 1,
                }
                buf = io.BytesIO()
                torch.save(checkpoint, buf)
                buf.seek(0)
                out_fs, _ = fsspec.core.url_to_fs(output_dir, **storage_options)
                out_fs.makedirs(output_dir, exist_ok=True)
                with out_fs.open(f"{output_dir}/best_model.pt", "wb") as f:
                    f.write(buf.read())
                print(f"  Saved best model (val_loss={best_val_loss:.4f}, val_acc={val_acc:.4f})")
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    if rank == 0:
                        print(f"  Early stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                    break

    if rank == 0:
        total_time = _time.time() - train_start
        peak_gpu_mem_gb = round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0
        if use_mlflow:
            try:
                with mlflow.start_span(name="model_save_and_register") as _sp:
                    _sp.set_inputs({"output_dir": output_dir})
                    _sp.set_outputs({"best_val_loss": best_val_loss, "total_time_s": round(total_time, 1),
                                     "peak_gpu_mem_gb": peak_gpu_mem_gb,
                                     "model_location": f"{output_dir}/best_model.pt"})
            except Exception:
                pass
        try:
            _fs, _ = fsspec.core.url_to_fs(output_dir, **storage_options)
            model_size_mb = round(_fs.info(f"{output_dir}/best_model.pt")["size"] / 1e6, 1)
        except Exception:
            model_size_mb = 0
        if use_mlflow:
            mlflow.log_metrics({
                "best_val_loss": best_val_loss,
                "best_val_acc": checkpoint.get("best_val_acc", 0) if best_val_loss < float("inf") else 0,
                "best_val_auc_roc": checkpoint.get("best_val_auc_roc", 0) if best_val_loss < float("inf") else 0,
                "best_val_f1": checkpoint.get("best_val_f1", 0) if best_val_loss < float("inf") else 0,
                "epochs_completed": checkpoint.get("epoch", 0) if best_val_loss < float("inf") else 0,
                "total_training_time_s": total_time,
                "avg_throughput_samples_per_sec": len(train_ds) * world_size * epochs / total_time,
                "peak_gpu_memory_gb": peak_gpu_mem_gb,
                "model_size_mb": model_size_mb,
            })
            try:
                import json as _json, tempfile as _tmpf, requests as _req
                _req.packages.urllib3.disable_warnings()
                _card = {
                    "model_name": "smartshop-rec-twotower",
                    "model_class": "TwoTower",
                    "framework": "pytorch",
                    "artifact_uri": f"{output_dir}/best_model.pt",
                    "architecture": {"n_users": n_users, "n_items": n_items,
                                     "embed_dim": embed_dim, "hidden_dim": hidden_dim, "dropout": 0.2},
                    "metrics": {"best_val_loss": best_val_loss,
                                "best_val_acc": checkpoint.get("best_val_acc", 0),
                                "best_val_auc_roc": checkpoint.get("best_val_auc_roc", 0),
                                "best_val_f1": checkpoint.get("best_val_f1", 0),
                                "epochs_completed": checkpoint.get("epoch", 0)},
                    "storage": {"format": "pytorch_state_dict",
                                "location": f"{output_dir}/best_model.pt"},
                }
                _run_id = mlflow.active_run().info.run_id
                _exp_id = mlflow.active_run().info.experiment_id
                _ws = workspace or "smartshop"
                _base = tracking_uri
                _tok = os.environ.get("MLFLOW_TRACKING_TOKEN", "")
                _art_url = f"{_base}/api/2.0/mlflow-artifacts/artifacts/workspaces/{_ws}/{_exp_id}/{_run_id}/artifacts/model/model_card.json"
                _payload = _json.dumps(_card, indent=2).encode()
                _r = _req.put(_art_url, data=_payload, headers={
                    "Authorization": f"Bearer {_tok}",
                    "X-MLflow-Workspace": _ws,
                    "Content-Type": "application/octet-stream",
                }, verify=False, timeout=30)
                if _r.status_code == 200:
                    print(f"  Logged model_card.json artifact to MLflow")
                else:
                    print(f"  Artifact upload returned {_r.status_code}: {_r.text}")
                mlflow.set_tag("model_s3_uri", f"{output_dir}/best_model.pt")
                mlflow.set_tag("model_format", "pytorch_state_dict")
                mlflow.set_tag("model_size_mb", str(model_size_mb))
                try:
                    mlflow.pyfunc.log_model(
                        artifact_path="model",
                        python_model=mlflow.pyfunc.PythonModel(),
                        registered_model_name="smartshop-rec-twotower",
                    )
                    print("  Logged model to MLflow Models tab")
                except Exception as _lme:
                    print(f"  log_model partial (non-fatal): {_lme}")
                from mlflow.tracking import MlflowClient
                _client = MlflowClient()
                try:
                    _client.create_registered_model("smartshop-rec-twotower",
                        tags={"task": "recommendation", "framework": "pytorch"},
                        description="Two-tower collaborative filtering model")
                except Exception:
                    pass
                _client.create_model_version(
                    name="smartshop-rec-twotower",
                    source=f"{output_dir}/best_model.pt",
                    run_id=_run_id,
                    description=f"val_acc={checkpoint.get('best_val_acc', 0):.4f}")
                print(f"  Registered model version: smartshop-rec-twotower")
            except Exception as _me:
                print(f"  Model registration failed (non-fatal): {_me}")
            _converged = patience_counter < patience
            mlflow.set_tag("convergence_status", "converged" if _converged else "early_stopped")
            mlflow.set_tag("training_status", "success")
            mlflow.set_tag("run_summary",
                           f"val_acc={checkpoint.get('best_val_acc',0):.4f} | "
                           f"AUC={checkpoint.get('best_val_auc_roc',0):.4f} | "
                           f"{round(total_time/60,1)}min | {round(throughput):,} samples/s")
            mlflow.end_run()
        print(f"Training complete in {total_time:.0f}s. Best val_loss: {best_val_loss:.4f}")
        print(f"Model saved to: {output_dir}/best_model.pt")

    dist.destroy_process_group()

print("train_fn defined")

## Submit TrainJob

In [ ]:
import base64
from datetime import datetime
from kubeflow.trainer import CustomTrainer
from kubeflow.trainer.options import (
    Name, Labels, PodTemplateOverrides, PodTemplateOverride,
    PodSpecOverride, ContainerOverride
)

# Read secrets from cluster to pass as env vars
v1 = k8s.CoreV1Api(k8s.ApiClient(cfg) if cfg else k8s.ApiClient())
s3_secret = v1.read_namespaced_secret(S3_CREDENTIALS_SECRET, NAMESPACE)
mlflow_secret = v1.read_namespaced_secret(MLFLOW_SECRET, NAMESPACE)

def decode_secret(secret, key):
    return base64.b64decode(secret.data[key]).decode()

job_id = datetime.now().strftime('%m%d-%H%M')
JOB_NAME = f"rec-train-{job_id}"

# Guard: skip if a rec-train job is already running or completed
_custom_api = k8s.CustomObjectsApi(k8s.ApiClient(cfg) if cfg else k8s.ApiClient())
try:
    _jobs = _custom_api.list_namespaced_custom_object(
        "trainer.kubeflow.org", "v1", NAMESPACE, "trainjobs"
    ).get("items", [])
    _active = [
        j["metadata"]["name"] for j in _jobs
        if j["metadata"]["name"].startswith("rec-train-")
        and not any(
            c.get("type") == "Failed" and c.get("status") == "True"
            for c in (j.get("status", {}).get("conditions") or [])
        )
    ]
except Exception:
    _active = []

if _active:
    JOB_NAME = _active[0]
    print(f"Active TrainJob already exists: {JOB_NAME} — skipping submission")
    _SKIP_SUBMIT = True
else:
    _SKIP_SUBMIT = False

func_args = {
    'data_dir': DATA_DIR,
    'output_dir': OUTPUT_DIR,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'lr': LR,
    'embed_dim': EMBED_DIM,
    'hidden_dim': HIDDEN_DIM,
    'max_rows': MAX_ROWS,
}

if not _SKIP_SUBMIT:
    job = trainer.train(
        trainer=CustomTrainer(
            func=train_fn,
            func_args=func_args,
            num_nodes=NUM_NODES,
            resources_per_node={'gpu': GPUS_PER_NODE, 'cpu': 8, 'memory': '32Gi'},
            env={
                'MLFLOW_TRACKING_URI': MLFLOW_TRACKING_URI,
                'MLFLOW_TRACKING_INSECURE_TLS': 'true',
                'MLFLOW_TRACKING_TOKEN': decode_secret(mlflow_secret, 'MLFLOW_TRACKING_TOKEN'),
                'MLFLOW_WORKSPACE': decode_secret(mlflow_secret, 'MLFLOW_WORKSPACE'),
                'AWS_ENDPOINT_URL_S3': MINIO_ENDPOINT,
                'S3_ENDPOINT': MINIO_ENDPOINT,
                'AWS_ACCESS_KEY_ID': decode_secret(s3_secret, 'AWS_ACCESS_KEY_ID'),
                'AWS_SECRET_ACCESS_KEY': decode_secret(s3_secret, 'AWS_SECRET_ACCESS_KEY'),
                'NCCL_DEBUG': 'INFO',
                'NCCL_IB_DISABLE': '1',
            },
        ),
        runtime=runtime,
        options=[
            Name(JOB_NAME),
            Labels({'app': 'smartshop', 'component': 'rec-training'}),
            PodTemplateOverrides(PodTemplateOverride(
                target_jobs=['node'],
                spec=PodSpecOverride(
                    volumes=[
                        {'name': 'shm', 'emptyDir': {'medium': 'Memory'}},
                    ],
                    containers=[ContainerOverride(
                        name='node',
                        volume_mounts=[
                            {'name': 'shm', 'mountPath': '/dev/shm'},
                        ],
                    )]
                )
            ))
        ]
    )
    print(f"Submitted: {JOB_NAME}")
else:
    print(f"Reusing existing job: {JOB_NAME}")

## Monitor Training

In [ ]:
print(f"Waiting for {JOB_NAME} to start running...")
trainer.wait_for_job_status(name=JOB_NAME, status={"Running"}, timeout=600)
print(f"{JOB_NAME} is Running")

print(f"Waiting for completion (timeout {TIMEOUT_SECONDS}s)...")
trainer.wait_for_job_status(name=JOB_NAME, status={"Complete", "Failed"}, timeout=TIMEOUT_SECONDS)

status = trainer.get_job(JOB_NAME)
print(f"Final status: {status.status}")

_failed = False
if hasattr(status, 'status') and hasattr(status.status, 'conditions'):
    conditions = {c.type: c.status for c in (status.status.conditions or [])}
    if conditions.get('Failed') == 'True':
        _failed = True
        print("TrainJob reported Failed — checking if model was actually saved...")
        try:
            os.environ['AWS_ENDPOINT_URL_S3'] = MINIO_ENDPOINT
            import fsspec as _fs
            _s3, _ = _fs.core.url_to_fs(OUTPUT_DIR, endpoint_url=MINIO_ENDPOINT)
            _files = _s3.ls(OUTPUT_DIR)
            if any('best_model.pt' in f for f in _files):
                print("Model artifact found on S3 — training succeeded despite exit barrier error.")
                _failed = False
            else:
                print("No model artifact found — genuine failure.")
        except Exception as _e:
            print(f"S3 check failed: {_e}")

if _failed:
    print("Last 30 lines of logs:")
    for line in list(trainer.get_job_logs(JOB_NAME, follow=False))[-30:]:
        print(line)
    raise RuntimeError(f"TrainJob {JOB_NAME} failed")

print(f"TrainJob {JOB_NAME} completed successfully!")

## Verify model artifact in S3

In [ ]:
os.environ['AWS_ENDPOINT_URL_S3'] = MINIO_ENDPOINT
import fsspec

fs, _ = fsspec.core.url_to_fs(OUTPUT_DIR, endpoint_url=MINIO_ENDPOINT)
files = fs.ls(OUTPUT_DIR)
print(f"Model artifacts in {OUTPUT_DIR}:")
for f in files:
    info = fs.info(f)
    size_mb = info.get('size', 0) / (1024 * 1024)
    print(f"  {os.path.basename(f):40s} {size_mb:>8.1f} MB")

assert any('best_model.pt' in f for f in files), 'best_model.pt not found!'
print('\nbest_model.pt found. Rec training PASSED.')

In [ ]:
print('NOTEBOOK_STATUS: SUCCESS')